In [0]:
dbutils.widgets.text("env","dev")

env=dbutils.widgets.get("env")

catalog=f"{env}_healthcare"

In [0]:
from pyspark.sql.functions import *

In [0]:
patients=spark.table(f"{catalog}.gold.dim_patient")

providers=spark.table(f"{catalog}.gold.dim_provider")

claims=spark.table(f"{catalog}.gold.fact_claims")

In [0]:
patients=patients.withColumn(
    "patient_age",
    floor(
        months_between(
            current_date(),
            col("birth_date")
        )/12
    )
)

In [0]:
feature_df=(
    claims.alias("c")
    .join(
        patients.alias("p"),
        "patient_id",
        "left"
    )
    .join(
        providers.alias("pr"),
        "provider_id",
        "left"
    )
)

In [0]:
windowSpec=Window.partitionBy("patient_id")

feature_df=feature_df.withColumn(
    "previous_claim_count",
    count("*").over(windowSpec)-1
)

In [0]:
feature_df=feature_df.withColumn(
    "risk_label",
    when(
        (
            col("amount")>5000
        )
        &
        (
            col("outstanding")>1000
        ),
        1
    ).otherwise(0)
)

In [0]:
feature_df=feature_df.withColumn(
    "risk_label",
    when(
        (
            col("amount")>5000
        )
        &
        (
            col("outstanding")>1000
        ),
        1
    ).otherwise(0)
)

In [0]:
ml_features=feature_df.select(

"claim_id",

"patient_id",

"provider_id",

"patient_age",

"gender",

"race",

"speciality",

"encounter_class",

"procedure_code",

"amount",

"payments",

"adjustments",

"outstanding",

"previous_claim_count",

"risk_label"
)

In [0]:
display(ml_features)